# SVG-Transformer — Demo aller drei Modi

**DE** — Dieses Notebook führt den Transformer anhand der Beispieldaten unter `examples/` vor.
Es erwartet, dass es aus dem Verzeichnis `notebooks/` heraus gestartet wird; alle Pfade sind
relativ zum Repository-Wurzelverzeichnis aufgelöst.

**EN** — This notebook demonstrates the transformer using the sample data under `examples/`.
It expects to be started from the `notebooks/` directory; all paths are resolved relative to
the repository root.

Beispieldaten / sample data: Jüdischer Friedhof Walsdorf (wld), *Steinerne Zeugen digital*,
CC BY-SA 4.0 — siehe / see `examples/LICENSE`.

## Vorbereitung / Setup

In [ ]:
import os
import sys

# Repository-Wurzel bestimmen und als Arbeitsverzeichnis setzen, damit die
# relativen Beispielpfade unabhängig vom Startort stimmen. /
# Determine the repository root and use it as the working directory so the
# relative sample paths hold regardless of where the notebook was started.
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

sys.path.insert(0, 'src')

from svg_transformer import (
    SVGTransformer, SVGTransformerError, StylesheetError, DXFError)

EXAMPLES = 'examples'
OUT = os.path.join(EXAMPLES, 'output')
os.makedirs(OUT, exist_ok=True)

print('Arbeitsverzeichnis / working directory:', os.getcwd())
print('Ausgabeverzeichnis / output directory: ', OUT)

## Modus 1 — `dxf`: Zeichnung in Plan umwandeln / turning the drawing into a plan

**DE** — Liest die DXF-Zeichnung und das Stylesheet, bestimmt den Referenzrahmen, ordnet den
Grabstätten-Polygonen ihre Bezeichnungen zu und schreibt SVG, JSON und CSV.
Die Zeile *Eindeutig zugeordnet* am Ende ist die wichtigste Kontrolle: Jedes Bezeichnungsfeld
sollte genau ein Polygon getroffen haben.

**EN** — Reads the DXF drawing and the stylesheet, determines the reference frame, assigns their
labels to the grave polygons and writes SVG, JSON and CSV. The line *Uniquely assigned* at the
end is the key check: every label field should have matched exactly one polygon.

In [ ]:
transformer = SVGTransformer(
    dxf_filepath=os.path.join(EXAMPLES, 'dxf', 'walsdorf.dxf'),
    stylesheet_filepath=os.path.join(
        EXAMPLES, 'stylesheets', 'wld-accurate_bb.xml'),
    svg_filepath=os.path.join(OUT, 'wld-accurate_bb.svg'),
    csv_filepath=os.path.join(OUT, 'walsdorf-objects.csv'),
    json_filepath=os.path.join(OUT, 'walsdorf-objects.json'),
    mode='dxf',
)

transformer.run()

In [ ]:
# Kurze Kontrolle des Ergebnisses / brief check of the result
from lxml import etree as et

root = et.parse(os.path.join(OUT, 'wld-accurate_bb.svg')).getroot()
gruppen = [g.get('id') for g in root.findall('{*}g')]
# Grabstätten-IDs enthalten einen Punkt (wld-01.002.003-00); die übrigen
# Elemente tragen fortlaufende IDs nach dem Muster wld-<gruppe>-<n>. /
# Grave IDs contain a dot; the remaining elements carry sequential IDs
# following the pattern wld-<group>-<n>.
grab_ids = root.xpath("//*[contains(@id, '.')]")
andere = root.xpath("//*[starts-with(@id, 'wld-') and not(contains(@id, '.'))]")

print('Gruppen im SVG / groups in the SVG:', gruppen)
print('Elemente mit Grabstätten-ID / elements with a grave ID:', len(grab_ids))
print('Beispiel-IDs / sample IDs:', [e.get('id') for e in grab_ids[:4]])
print('Übrige Elemente / remaining elements:', len(andere),
      [e.get('id') for e in andere[:3]])

### Ergebnis ansehen / viewing the result

**DE** — Das erzeugte SVG lässt sich direkt im Notebook anzeigen. Die dunkle Variante (`bb`)
hat einen transparenten Hintergrund mit weißen Linien und wirkt auf hellem Grund blass —
hier deshalb die helle Variante aus Modus 2.

**EN** — The generated SVG can be displayed directly in the notebook. The dark variant (`bb`)
has a transparent background with white lines and looks pale on a light background — the light
variant from mode 2 is shown instead.

## Modus 2 — `json`: Stylesheet wechseln, Geometrie behalten / swapping the stylesheet

**DE** — Modus 1 hat die Geometrie in eine JSON-Datei gesichert. Damit lässt sich ein neues SVG
erzeugen, ohne die DXF erneut zu lesen — deutlich schneller und sinnvoll, wenn nur Farben,
Strichstärken oder der Plankopf geändert werden.

**EN** — Mode 1 stored the geometry in a JSON file. This allows generating a new SVG without
reading the DXF again — considerably faster and useful when only colours, stroke widths or the
title block change.

**DE** — Hier wird auf die helle Marker-Variante gewechselt: dieselben Grabstätten, aber als
Kreissymbole auf weißem Grund.

**EN** — Here we switch to the light marker variant: the same graves, but as circle symbols on a
white background.

In [ ]:
transformer = SVGTransformer(
    stylesheet_filepath=os.path.join(
        EXAMPLES, 'stylesheets', 'wld-marker_wb.xml'),
    json_filepath=os.path.join(OUT, 'walsdorf-objects.json'),
    svg_filepath=os.path.join(OUT, 'wld-marker_wb.svg'),
    mode='json',
)

transformer.run()

In [ ]:
from IPython.display import SVG, display

display(SVG(filename=os.path.join(OUT, 'wld-marker_wb.svg')))

## Modus 3 — `mapping`: Kartierung / thematic mapping

**DE** — Der fertige Plan wird anhand einer CSV eingefärbt. Kartiert wird die Datierung der
Grabsteine als Farbverlauf; zusätzlich erhält jeder Stein einen Tooltip und einen Link in die
epidat-Datenbank des Salomon-Ludwig-Steinheim-Instituts.

**EN** — The finished plan is colour-coded from a CSV. The dating of the grave stones is mapped
as a colour gradient; in addition, each stone receives a tooltip and a link into the epidat
database of the Salomon Ludwig Steinheim Institute.

In [ ]:
import pandas as pd

df = pd.read_csv(os.path.join(EXAMPLES, 'data', 'wld-belegung.csv'), sep=';')
print('Zeilen / rows:', len(df))
print('Datierung von / dating from', int(df['Datierung'].min()),
      'bis / to', int(df['Datierung'].max()))
df.head()

In [ ]:
transformer = SVGTransformer(mode='mapping')

# Als Eingabe dient der in Modus 2 erzeugte Plan. Alternativ liegt unter
# examples/prepared/ eine mitgelieferte Fassung. /
# The plan produced in mode 2 serves as input. Alternatively, a supplied
# version is available under examples/prepared/.
transformer.load_svg(os.path.join(OUT, 'wld-marker_wb.svg'))
transformer.load_csv(
    os.path.join(EXAMPLES, 'data', 'wld-belegung.csv'),
    dtype={'SZd-ID': str, 'Datierung': int},
)

gradient = transformer.generate_gradient_color_table(
    min_val=1630, max_val=1920, cmap_name='jet', steps=290)

print('Farbtabelle / colour table:', len(gradient), 'Einträge / entries')
print('1630 →', gradient[1630], '| 1920 →', gradient[1920])

In [ ]:
transformer.apply_mapping(
    id_col='SZd-ID',
    value_col='Datierung',
    color_table=gradient,
    tooltip_cols=['SZd-ID', 'Sterbejahr', 'epidat'],
    url_col='Weblink',
    require_tooltip_if_any=True,
)

transformer.save_svg(os.path.join(OUT, 'wld-belegung.svg'))

In [ ]:
display(SVG(filename=os.path.join(OUT, 'wld-belegung.svg')))

**DE** — Im Browser geöffnet zeigt diese Datei beim Überfahren eines Grabsteins einen Tooltip
und führt per Klick auf den zugehörigen epidat-Eintrag. Im Notebook bleiben Tooltips und Links
wirkungslos, weil das eingebettete SVG kein JavaScript ausführt.

**EN** — Opened in a browser, this file shows a tooltip when hovering over a grave stone and
links to the corresponding epidat entry on click. Within the notebook, tooltips and links stay
inactive because the embedded SVG does not execute JavaScript.

## Fehlerbehandlung / error handling

**DE** — Fehlerhafte Eingaben lösen Ausnahmen mit zweisprachiger Meldung aus, statt den Prozess
zu beenden. Der Kernel bleibt dabei am Leben.

**EN** — Faulty input raises exceptions carrying a bilingual message instead of terminating the
process. The kernel stays alive.

In [ ]:
try:
    SVGTransformer(
        stylesheet_filepath='examples/stylesheets/gibt-es-nicht.xml',
        mode='dxf',
    ).parse_stylesheet()
except StylesheetError as e:
    print(type(e).__name__)
    print(e)

## Stylesheets validieren / validating the stylesheets

**DE** — `schema/stylesheet.xsd` ist XML Schema 1.0 und mit lxml prüfbar. Die strengeren,
bedingten Regeln in `schema/stylesheet-1.1.xsd` brauchen einen XSD-1.1-Validierer, etwa das
Paket `xmlschema` oder oXygen.

**EN** — `schema/stylesheet.xsd` is XML Schema 1.0 and can be checked with lxml. The stricter,
conditional rules in `schema/stylesheet-1.1.xsd` require an XSD 1.1 validator, such as the
`xmlschema` package or oXygen.

In [ ]:
import glob

xsd = et.XMLSchema(et.parse('schema/stylesheet.xsd'))
for f in sorted(glob.glob('examples/stylesheets/*.xml')):
    print(f'{os.path.basename(f):24s}', xsd.validate(et.parse(f)))

In [ ]:
# Optional: strenge Prüfung / optional: strict checking
# pip install xmlschema
#
# import xmlschema
# xsd11 = xmlschema.XMLSchema11('schema/stylesheet-1.1.xsd')
# for f in sorted(glob.glob('examples/stylesheets/*.xml')):
#     print(f'{os.path.basename(f):24s}', xsd11.is_valid(f))

---

**DE** — Erzeugte Dateien liegen in `examples/output/` und sind über `.gitignore` von der
Versionierung ausgenommen.

**EN** — Generated files are placed in `examples/output/` and excluded from versioning via
`.gitignore`.